# Biohub — Cell Tracking · ver 3 (phân bào theo profile độ sáng)

Ver 3 = ver 2 + tri thức từ [discussion #740573](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development/discussion/740573)
(đo trên 73 file nhãn train thật):

- **Sister separation** là tín hiệu hình học mạnh nhất: median 8,85 · p99 13,9 · **max 14,65 µm** → `DIV_SIBLING_GATE` 12 → **14,5 µm** (bắt thêm ~10% cặp chị em)
- **Khoảng cách mẹ→con KHÔNG phải tín hiệu** (base rate 24:1) → `DIV_PARENT_GATE` chỉ là cửa sổ tìm kiếm, 10 → 12 µm (bắt p99)
- **Mẹ sáng dần trước khi chia** (peak intensity AUC 0,73; anaphase "đường sáng mảnh" vài khung TRƯỚC tách) → ứng viên có mẹ sáng dần được ưu tiên
- **Ổn định khối lượng mẹ**: blob "mẹ" tại khung tách ≥ 1,7× baseline riêng = blob gộp 2 tế bào (merge-split ≈ 2×) → từ chối; mẹ thật chỉ sáng lên nhẹ 1,1–1,5×
- **Con kế thừa vận tốc mẹ** (bug fix: vectơ mẹ→con làm ứng viên chết "lost" mỗi khung)
- Kiểm chứng synthetic: **21/21 ĐẠT** — ver 3 chặn merge-split khéo mà ver 2 xác nhận nhầm, bắt chị-em-xa 13,4 µm mà ver 2 bỏ sót

Cách dùng: **File → Import Notebook** vào Kaggle, Add Input competition, Save & Run All (≥ 12h GPU không cần), Submit `submission.csv`.

In [ ]:
# ver 3 · cell 1 — IMPORTS
# Dán đè Cell 1 của notebook Kaggle "Biohub - Cell Tracking During Development".
# ============================================================

import itertools
import json
import os
import time

import blosc2
import numpy as np
import pandas as pd
from scipy.ndimage import center_of_mass, label, maximum_filter, uniform_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist


In [ ]:
# ver 3 · cell 2 — SIÊU THAM SỐ (mọi núm tune nằm ở đây)
# Dán đè Cell 2 của notebook Kaggle.
# ============================================================
# Nguyên tắc ver 3 (rút từ discussion #740573 — đo trên 73 file nhãn TRAIN
# THẬT: 36 phân bào có nhãn, 25.661 cạnh continuation, 572 track):
#   1. KHOẢNG CÁCH MẸ→CON KHÔNG PHẢI TÍN HIỆU PHÂN BÀO: IQR bước phân bào
#      (3,3–6,2µm) nằm gần trọn trong vùng bước thường (p99 = 6,9). Gate
#      ≥5µm thu 722 bước thường / 30 bước phân bào = 24:1 base rate.
#      → khoảng cách chỉ còn là CỬA SỔ TÌM KIẾM; tín hiệu thật là:
#      sister separation + bảo toàn độ sáng + ĐỘNG HỌC + APPEARANCE.
#   2. SISTER SEPARATION là tín hiệu hình học mạnh nhất (median 8,85,
#      IQR 7,2–10,2, p99 13,9, MAX 14,65µm) → cổng 12 bỏ lỡ ~10% cặp.
#   3. MẸ SÁNG LÊN TRƯỚC KHI CHIA (peak intensity AUC 0,73 với tế bào
#      thường; hengck23: đường sáng mảnh anaphase xuất hiện VÀI KHUNG
#      TRƯỚC khung tách) → dùng làm (a) ưu tiên ứng viên, (b) chặn
#      merge-split qua "ổn định khối lượng mẹ": blob gộp 2 tế bào ≈ 2×
#      khối lượng đơn, còn mẹ thật chỉ sáng lên nhẹ (1,1–1,5×).
#   4. Mọi cạnh GT nối đúng 1 khung (25.661/25.661) → nội suy node tại
#      khung mất vẫn là bắt buộc (giữ từ ver 2).
#   5. GT chỉ gắn nhãn theo segment (median 35 khung/track) → phát hiện
#      phân bào nơi GT không nhãn vẫn tính FP → precision > recall.
# ============================================================

TEST_DIR = '/kaggle/input/competitions/biohub-cell-tracking-during-development/test'

# Thang vật lý (Z, Y, X) — µm/voxel, theo đề bài
SCALE = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)

# --- Tầng 1 · DETECTION (giữ nguyên ver 2) ---
DS_Z, DS_Y, DS_X = 2, 4, 4       # downsample bất đối xứng: z giữ kỹ hơn
SMOOTH_SIZE = 3                   # uniform_filter, trong không gian downsample
PERCENTILE = 90.0                 # bắt thêm nhân mờ (recall > precision)
MIN_NVOXELS = 4                   # bỏ component li ti (nhiễu)
MAX_NVOXELS = 3000                # blob gộp vẫn nhận, tách ở tầng 1b
CONN26 = True                     # liên kết 26-ô: chống tách nhân giữa các lát z

# --- Tầng 1b · TÁCH BLOB GỘP (giữ nguyên ver 2) ---
SPLIT_MIN_NVOXELS = 90            # component ds lớn hơn mới xét tách
PEAK_SIZE = (3, 5, 5)             # maximum_filter tìm đỉnh cục bộ (z,y,x ds)
MIN_PEAK_DIST_DS = 4.0            # 2 đỉnh cách ≥ 4 voxel ds
PEAK_MIN_BRIGHT = 1.15            # đỉnh sáng hơn ngưỡng ít nhất 15%

# --- Tầng 2 · LINKING (giữ nguyên ver 2) ---
BASE_GATE_UM = 9.0
GATE_MEDIAN_MULT = 2.5
GATE_MIN_UM, GATE_MAX_UM = 5.0, 14.0
VEL_SMOOTH = 0.5                  # vận tốc EMA: v ← 0.5·mới + 0.5·cũ
BRIGHT_WEIGHT = 0.0               # phạt chênh độ sáng trong cost (tắt)

# --- Frame-skip + nội suy (giữ nguyên ver 2) ---
ALLOW_FRAME_SKIP = True
SKIP_GATE_UM = 12.0
MAX_SKIP_FRAMES = 2
INTERPOLATE_MISSED_FRAMES = True  # GT 100% cạnh liền khung → nội suy bắt buộc
TIME_LIMIT_HOURS = 11.0

# --- Tầng 3 · PHÂN BÀO THEO PROFILE ĐỘ SÁNG (mới trong ver 3) ---
DIVISION_ENABLED = True
# Khoảng cách = cửa sổ tìm kiếm (base rate 24:1 — không phải tín hiệu):
DIV_PARENT_GATE_UM = 12.0         # 10 → 12: bắt p99 bước mẹ→con (10,6, max 12,3)
DIV_SIBLING_GATE_UM = 14.5       # 12 → 14,5: p99 sister sep 13,9, max 14,65
DIV_MIN_CHILD_FRAC = 0.15         # con thứ 2 phải ≥ 15% độ sáng mẹ
DIV_BRIGHTNESS_CHECK = True      # bảo toàn độ sáng: I(con1)+I(con2) ≈ I(mẹ)
DIV_BRIGHTNESS_RATIO = (0.55, 1.8)
DIV_CONFIRM_FRAMES = 3           # 2 con phải sống ≥ 3 khung sau khi "sinh"
DIV_SEP_GROWTH = 1.15            # khoảng cách 2 con tăng ≥ 15% sau 3 khung
# ... ver 3 mới: theo dõi khối lượng từng track qua các khung —
MASS_HISTORY = 8                  # số khung gần nhất đưa vào baseline
MASS_SKIP_LAST = 2                # loại 2 khung cuối (đúng lúc mẹ sáng lên)
DIV_MOM_MAX_RISE = 1.7           # mẹ tại khung tách ≤ 1,7× baseline riêng nó
                                 # (blob gộp 2 tế bào ≈ 2×; mẹ thật 1,1–1,5×)
DIV_MOM_MIN_FRAC = 0.5           # mẹ phai quá (đetection lép) → nghi merge-split
DIV_MOM_BRIGHT_BONUS = 1.05      # ưu tiên ứng viên có mẹ sáng dần ≥ 5%
MASS_BASE_MIN_FRAMES = 4         # track non hơn → chưa đủ baseline, bỏ qua check

# --- Soát bằng mắt (vẽ 1 khung đầu tiên) ---
RUN_PREVIEW = True

# --- Chẩn đoán ---
DIAGNOSE = True

NODE_ID = itertools.count(1)      # node_id duy nhất toàn cục
BIG = 1e9


In [ ]:
# ver 3 · cell 3 — PIPELINE: ĐỌC ZARR → DETECTION (+TÁCH BLOB) → TRACKING
#                          → PHÂN BÀO THEO PROFILE ĐỘ SÁNG
# Dán đè Cell 3 của notebook Kaggle (toàn bộ thuật toán nằm ở đây).
# ============================================================
# Thay đổi so với ver 2 (nguồn: discussion #740573, đo trên nhãn train):
#   * Mỗi track theo dõi LỊCH SỬ KHỐI LƯỢNG qua các khung (mass_hist).
#   * Ứng viên phân bào bị chặn nếu blob "mẹ" tại khung tách ≥ 1,7× baseline
#     riêng của nó — blob gộp 2 tế bào (merge-split) có khối lượng ≈ 2× đơn,
#     còn mẹ thật chỉ sáng lên nhẹ trước khi chia (peak AUC 0,73).
#   * Ứng viên có mẹ sáng dần (≥ 1,05× baseline) được XẾP TRƯỚC khi cạnh
#     tranh nhau con thứ 2 — tín hiệu appearance mạnh hơn hình học.
#   * DIV_SIBLING_GATE 14,5µm (p99 thật 13,9, max 14,65) — bắt thêm cặp
#     chị em xa mà ver 2 bỏ sót; các merge-split thừa lọt qua cửa rộng này
#     sẽ bị mass stability chặn lại.
#   * DIV_PARENT_GATE 12µm — khoảng cách chỉ là cửa sổ tìm kiếm (base rate
#     24:1), không còn là tín hiệu.
# ============================================================

STRUCT26 = np.ones((3, 3, 3), dtype=bool)
T_START = time.time()


def _over_time_budget():
    return (time.time() - T_START) > TIME_LIMIT_HOURS * 3600.0


# ---------- 1) ĐỌC DỮ LIỆU (giữ nguyên ver 2) ----------
def read_zarr_meta(zarr_path):
    """Đọc shape/dtype/chunk grid từ zarr.json của mảng (như notebook gốc)."""
    with open(os.path.join(zarr_path, '0', 'zarr.json')) as f:
        meta = json.load(f)
    shape = tuple(int(v) for v in meta['shape'])            # (T, Z, Y, X)
    dtype = np.dtype(meta['data_type'])
    chunk = meta.get('chunk_grid', {}).get('configuration', {}).get('chunk_shape')
    if not chunk or len(chunk) != 4:
        chunk = [1, shape[1], shape[2], shape[3]]
    return shape, dtype, [int(c) for c in chunk]


def load_volume(zarr_path, t, shape, dtype, chunk):
    """Đọc khung t → (Z, Y, X). Chunk 0/0/0 là đường nhanh như notebook gốc;
    nếu khung được chia nhiều chunk thì lắp ghép đủ các chunk có mặt."""
    T, Z, Y, X = shape
    _, cZ, cY, cX = chunk
    croot = os.path.join(zarr_path, '0', 'c', str(t))
    if not os.path.isdir(croot):
        raise FileNotFoundError(f'không tìm thấy khung t={t}: {croot}')

    def read_chunk(cz, cy, cx):
        p = os.path.join(croot, str(cz), str(cy), str(cx))
        with open(p, 'rb') as f:
            return np.frombuffer(blosc2.decompress(f.read()), dtype=dtype)

    # Đường nhanh: metadata nói 1 chunk chứa cả khung
    if (cZ, cY, cX) == (Z, Y, X):
        try:
            arr = read_chunk(0, 0, 0)
            if arr.size == Z * Y * X:
                return arr.reshape(Z, Y, X)
        except Exception:
            pass  # rơi xuống đường đa chunk bên dưới

    vol = np.zeros((Z, Y, X), dtype=dtype)
    placed = 0
    for cz in range((Z + cZ - 1) // cZ):
        for cy in range((Y + cY - 1) // cY):
            for cx in range((X + cX - 1) // cX):
                p = os.path.join(croot, str(cz), str(cy), str(cx))
                if not os.path.isfile(p):
                    continue
                try:
                    arr = read_chunk(cz, cy, cx)
                except Exception:
                    continue
                vz = min(cZ, Z - cz * cZ)
                vy = min(cY, Y - cy * cY)
                vx = min(cX, X - cx * cX)
                if vz <= 0 or vy <= 0 or vx <= 0:
                    continue
                if arr.size >= cZ * cY * cX:      # chunk đầy (zarr v3 pad theo fill)
                    block = arr[:cZ * cY * cX].reshape(cZ, cY, cX)[:vz, :vy, :vx]
                elif arr.size >= vz * vy * vx:     # biến thể chunk gọn ở biên
                    block = arr[:vz * vy * vx].reshape(vz, vy, vx)
                else:
                    continue
                vol[cz * cZ:cz * cZ + vz, cy * cY:cy * cY + vy, cx * cX:cx * cX + vx] = block
                placed += 1
    if placed == 0:
        raise FileNotFoundError(f'khung t={t}: không đọc được chunk nào từ {croot}')
    return vol


# ---------- 2) DETECTION + TÁCH BLOB GỘP (giữ nguyên ver 2) ----------
def _com(coords, w):
    """Center-of-mass theo cường độ của tập voxel (z,y,x ds, float)."""
    s = w.sum()
    if s <= 0:
        return coords.mean(axis=0)
    return (coords * w[:, None]).sum(axis=0) / s


def detect_nodes(vol):
    """Trả về list dict: z, y, x (voxel gốc, int — đúng format đề bài) và
    zf, yf, xf (float — cho tracking), mass (tổng cường độ), nvox."""
    ds = vol[::DS_Z, ::DS_Y, ::DS_X].astype(np.float32)
    if ds.size == 0:
        return []
    smoothed = uniform_filter(ds, size=SMOOTH_SIZE)
    thr = np.percentile(smoothed, PERCENTILE)
    mask = smoothed > thr
    labeled, n = label(mask, structure=STRUCT26 if CONN26 else None)
    if n == 0:
        return []

    # đỉnh cục bộ (tính 1 lần cả khung): voxel = max trong lân cận PEAK_SIZE
    # và sáng hơn ngưỡng ít nhất PEAK_MIN_BRIGHT (chống đỉnh nhiễu)
    mf = maximum_filter(smoothed, size=PEAK_SIZE)
    is_peak = (smoothed == mf) & mask & (smoothed >= thr * PEAK_MIN_BRIGHT)

    lab_flat = labeled.ravel()
    counts = np.bincount(lab_flat, minlength=n + 1)
    wsums = np.bincount(lab_flat, weights=smoothed.ravel(), minlength=n + 1).astype(np.float64)
    coms = center_of_mass(smoothed, labeled, index=range(1, n + 1))

    # trọng số khoảng cách trong không gian ds: z gấp đôi (bước z dài gấp đôi)
    ZW = np.array([2.0, 1.0, 1.0])

    nodes = []
    for i in range(1, n + 1):
        nv = int(counts[i])
        if nv < MIN_NVOXELS or nv > MAX_NVOXELS:
            continue

        split_done = False
        if nv > SPLIT_MIN_NVOXELS:
            comp = np.argwhere(labeled == i)                # (N, 3) z,y,x
            pk_mask = is_peak[comp[:, 0], comp[:, 1], comp[:, 2]]
            peaks = comp[pk_mask]
            if len(peaks) >= 2:
                # giữ các đỉnh sáng nhất, bỏ đỉnh quá gần một đỉnh đã giữ
                vals = smoothed[peaks[:, 0], peaks[:, 1], peaks[:, 2]]
                order = np.argsort(-vals)
                kept = []
                for idx in order:
                    p = peaks[idx].astype(np.float64)
                    too_close = False
                    for q in kept:
                        if np.linalg.norm((p - q) * ZW) < MIN_PEAK_DIST_DS:
                            too_close = True
                            break
                    if not too_close:
                        kept.append(p)
                if len(kept) >= 2:
                    # gán mỗi voxel component → đỉnh gần nhất; CoM riêng từng vùng
                    P = np.array(kept)
                    D = cdist(comp.astype(np.float64) * ZW, P * ZW)
                    assign = D.argmin(axis=1)
                    w_all = smoothed[comp[:, 0], comp[:, 1], comp[:, 2]].astype(np.float64)
                    for k in range(len(kept)):
                        sel = assign == k
                        sub = comp[sel]
                        w = w_all[sel]
                        if sub.shape[0] < MIN_NVOXELS:
                            continue
                        cz, cy, cx = _com(sub.astype(np.float64), w)
                        nodes.append({
                            'z': int(round(cz * DS_Z)), 'y': int(round(cy * DS_Y)),
                            'x': int(round(cx * DS_X)),
                            'zf': cz * DS_Z, 'yf': cy * DS_Y, 'xf': cx * DS_X,
                            'mass': float(w.sum()), 'nvox': int(sub.shape[0]),
                        })
                    split_done = True
        if split_done:
            continue

        cz, cy, cx = coms[i - 1]
        nodes.append({
            'z': int(round(cz * DS_Z)), 'y': int(round(cy * DS_Y)), 'x': int(round(cx * DS_X)),
            'zf': cz * DS_Z, 'yf': cy * DS_Y, 'xf': cx * DS_X,
            'mass': float(wsums[i]), 'nvox': nv,
        })
    return nodes


# ---------- 3) TRACKER (ver 3: profile độ sáng cho phân bào) ----------
def _mass_baseline(hist):
    """Baseline khối lượng của track: trung vị MASS_HISTORY giá trị gần nhất,
    BỎ MASS_SKIP_LAST khung cuối (đúng khoảng mẹ sáng lên trước khi chia).
    Trả về None nếu chưa đủ dữ liệu."""
    if len(hist) < MASS_BASE_MIN_FRAMES + MASS_SKIP_LAST:
        return None
    kept = hist[-(MASS_HISTORY + MASS_SKIP_LAST):-MASS_SKIP_LAST]
    if not kept:
        return None
    return float(np.median(kept))


class Tracker:
    """Hungarian + motion model + tách blob + PHÂN BÀO THEO PROFILE ĐỘ SÁNG.

    Mỗi khung gồm 5 bước:
      (a) Hungarian giữa node khung trước (đặt tại vị trí DỰ ĐOÁN) và node
          khung hiện tại, gate thích ứng
      (b) phân bào ỨNG VIÊN — 5 CỬA:
          1. parent gate 12µm (cửa sổ tìm kiếm — base rate 24:1 nên khoảng
             cách không còn là tín hiệu)
          2. sibling gate 14,5µm (p99 thật 13,9 — bắt cả cặp chị em xa)
          3. bảo toàn độ sáng (con1+con2 ≈ mẹ) + con thứ 2 ≥ 15% mẹ
          4. MỚI — ỔN ĐỊNH KHỐI LƯỢNG MẸ: khối lượng blob mẹ tại khung tách
             ≤ 1,7× baseline riêng của nó (blob gộp 2 tế bào ≈ 2×; mẹ thật
             chỉ sáng lên nhẹ) và ≥ 0,5× (mẹ không phai đột ngột)
          5. MỚI — Ưu tiên appearance: ứng viên có mẹ sáng dần ≥ 5% xếp trước
          Cạnh divergence vẫn HOÃN chờ bước (e)
      (c) frame-skip: nối lại track mất ≤ 2 khung + nội suy node giữa
      (d) dọn dẹp
      (e) xác nhận động học: 2 con sống ≥ DIV_CONFIRM_FRAMES khung và
          khoảng cách tăng ≥ 15% — cạnh + division chỉ ghi khi vượt hết
    """

    def __init__(self):
        self.active = {}        # nid → dict(pos, vel, mass, mass_hist)
        self.pending = {}       # nid → dict(pos, vel, mass, mass_hist, missed)
        self.recent_steps = []  # bước đi (µm) của các match gần đây
        self.div_pending = []   # ứng viên phân bào chờ xác nhận
        self.n_div_confirmed = 0
        self.n_div_rejected = 0
        # ver 3: đếm lý do từ chối để chẩn đoán
        self.n_rej_mass = 0        # rớt ổn định khối lượng mẹ (bước b — MỚI)
        self.n_rej_lost = 0        # một con biến mất (bước e)
        self.n_rej_dyn = 0         # không tách ra đủ (bước e)

    def _gate(self):
        if len(self.recent_steps) < 5:
            return BASE_GATE_UM
        med = float(np.median(self.recent_steps))
        return float(np.clip(GATE_MEDIAN_MULT * med, GATE_MIN_UM, GATE_MAX_UM))

    @staticmethod
    def _push_hist(hist, m):
        h = list(hist)
        h.append(float(m))
        if len(h) > MASS_HISTORY + MASS_SKIP_LAST + 2:
            del h[:len(h) - (MASS_HISTORY + MASS_SKIP_LAST + 2)]
        return h

    def step(self, dets, t):
        """Xử lý khung t. Trả về (nodes, edges, divisions, interp_nodes)."""
        nodes = [(next(NODE_ID), d) for d in dets]
        ids = [nid for nid, _ in nodes]
        pos = np.array(
            [[d['zf'], d['yf'], d['xf']] for _, d in nodes], dtype=np.float64,
        ).reshape(-1, 3) * SCALE
        mass = [float(d['mass']) for _, d in nodes]

        edges, divisions, interp_nodes = [], [], []
        matched_curr = {}   # curr_idx → prev nid
        succ = {}           # prev nid → nid khung này (nối mạch cho bước (e))

        # ---- (a) matching chính: Hungarian trên vị trí DỰ ĐOÁN ----
        prev_ids = list(self.active.keys())
        if prev_ids and ids:
            pred = np.array(
                [self.active[p]['pos'] + self.active[p]['vel'] for p in prev_ids],
            )
            D = np.linalg.norm(pred[:, None, :] - pos[None, :, :], axis=2)
            gate = self._gate()
            cost = D
            if BRIGHT_WEIGHT > 0:
                mp = np.maximum(
                    np.array([self.active[p]['mass'] for p in prev_ids])[:, None], 1e-9)
                mc = np.maximum(np.array(mass)[None, :], 1e-9)
                cost = D * (1.0 + BRIGHT_WEIGHT * np.abs(np.log(mc / mp)))
            cost = np.where(D <= gate, cost, BIG)
            ri, ci = linear_sum_assignment(cost)
            for r, c in zip(ri, ci):
                if D[r, c] > gate:
                    continue
                p = prev_ids[r]
                matched_curr[c] = p
                succ[p] = ids[c]
                edges.append((p, ids[c]))
                self.recent_steps.append(
                    float(np.linalg.norm(pos[c] - self.active[p]['pos'])))
            if len(self.recent_steps) > 200:
                del self.recent_steps[:100]

        new_active = {}
        for c, p in matched_curr.items():
            a = self.active[p]
            disp = pos[c] - a['pos']
            vel = VEL_SMOOTH * disp + (1.0 - VEL_SMOOTH) * a['vel']
            # ver 3: lịch sử khối lượng đi THEO TRACK (chuyển từ nid cũ sang nid mới)
            new_active[ids[c]] = {
                'pos': pos[c], 'vel': vel, 'mass': mass[c],
                'mass_hist': self._push_hist(a['mass_hist'], mass[c]),
            }

        # ---- (b) phân bào ỨNG VIÊN (5 cửa — cạnh hoãn chờ bước (e)) ----
        unmatched = [j for j in range(len(ids)) if j not in matched_curr]
        if DIVISION_ENABLED and matched_curr:
            cands = []
            A_vel_cache = {p: self.active[p]['vel'] for p in matched_curr.values()}
            for c, p in matched_curr.items():
                A = self.active[p]
                P, mP, hist = A['pos'], A['mass'], A['mass_hist']
                C1, mC1 = pos[c], mass[c]
                base = _mass_baseline(hist)          # None nếu track quá non
                for j in unmatched:
                    B2, mB2 = pos[j], mass[j]
                    dP = float(np.linalg.norm(P - B2))
                    if dP > DIV_PARENT_GATE_UM:
                        continue
                    dS = float(np.linalg.norm(C1 - B2))
                    if dS > DIV_SIBLING_GATE_UM:
                        continue
                    if mP > 0:
                        if mB2 < DIV_MIN_CHILD_FRAC * mP:
                            continue
                        if DIV_BRIGHTNESS_CHECK:
                            ratio = (mC1 + mB2) / mP
                            if not (DIV_BRIGHTNESS_RATIO[0] <= ratio <= DIV_BRIGHTNESS_RATIO[1]):
                                continue
                    # --- MỚI ver 3: cửa 4 — ổn định khối lượng mẹ ---
                    rise = (mP / base) if (base and base > 0) else None
                    if rise is not None:
                        if rise > DIV_MOM_MAX_RISE:
                            # blob "mẹ" gộp ~2 tế bào → merge-split giả
                            self.n_rej_mass += 1
                            continue
                        if rise < DIV_MOM_MIN_FRAC:
                            # mẹ phai đột ngột → detection không ổn định
                            self.n_rej_mass += 1
                            continue
                    # --- MỚI ver 3: cửa 5 (soft) — ưu tiên mẹ sáng dần ---
                    bright = (rise is not None and rise >= DIV_MOM_BRIGHT_BONUS)
                    cands.append((0 if bright else 1, dP, p, c, j, rise))
            # sort: (mẹ sáng dần trước, rồi tới khoảng cách)
            cands.sort()
            used_p, used_j = set(), set()
            for _prio, dP, p, c, j, rise in cands:
                if p in used_p or j in used_j:
                    continue
                used_p.add(p)
                used_j.add(j)
                # con thứ 2 khởi động như track mới; cạnh p→ids[j] HOÃN,
                # chỉ ghi nếu được xác nhận ở bước (e) của các khung sau
                # QUAN TRỌNG: con KẾ THỪA vận tốc mẹ — không phải vectơ
                # mẹ→con! (vectơ đó làm dự đoán khung sau vọt xa vị trí thật
                # → con không match được → ứng viên chết "lost" và bị đề
                # xuất lại mỗi khung — vòng lặp đã bắt gặp khi kiểm chứng)
                new_active[ids[j]] = {
                    'pos': pos[j], 'vel': np.array(A_vel_cache[p]), 'mass': mass[j],
                    'mass_hist': [mass[j]],
                }
                self.div_pending.append({
                    't': t, 'mother': p, 'mother_mass': self.active[p]['mass'],
                    'mother_rise': rise,
                    'c1': ids[c], 'c2': ids[j], 'c2_start': ids[j],
                    'd0': float(np.linalg.norm(pos[c] - pos[j])), 'age': 0,
                })
            unmatched = [j for j in unmatched if j not in used_j]

        # ---- (c) frame-skip: nối lại track mất ≤ MAX_SKIP_FRAMES khung ----
        if ALLOW_FRAME_SKIP and self.pending and unmatched:
            pend_ids = list(self.pending.keys())
            pred = np.array([
                self.pending[q]['pos'] + self.pending[q]['vel'] * (self.pending[q]['missed'] + 1)
                for q in pend_ids
            ])
            C = pos[np.array(unmatched, dtype=int)]
            D = np.linalg.norm(pred[:, None, :] - C[None, :, :], axis=2)
            cost = np.where(D <= SKIP_GATE_UM, D, BIG)
            ri, ci = linear_sum_assignment(cost)
            for r, c in zip(ri, ci):
                if D[r, c] > SKIP_GATE_UM:
                    continue
                q, j = pend_ids[r], unmatched[c]
                Q = self.pending[q]
                gap = Q['missed'] + 1
                if INTERPOLATE_MISSED_FRAMES and gap >= 2:
                    # Chèn node nội suy tại các khung bị mất rồi nối các cạnh
                    # LIỀN KHUNG — cạnh nhảy t→t+2 bị metric bỏ hẳn.
                    chain = q
                    for k in range(1, gap):
                        pm = Q['pos'] + (pos[j] - Q['pos']) * (k / gap)  # µm
                        vf = pm / SCALE
                        mid = next(NODE_ID)
                        interp_nodes.append((mid, t - gap + k, {
                            'z': int(round(vf[0])), 'y': int(round(vf[1])), 'x': int(round(vf[2])),
                            'zf': float(vf[0]), 'yf': float(vf[1]), 'xf': float(vf[2]),
                            'mass': (Q['mass'] + mass[j]) / 2.0, 'nvox': 0,
                        }))
                        edges.append((chain, mid))
                        chain = mid
                    edges.append((chain, ids[j]))
                new_active[ids[j]] = {
                    'pos': pos[j],
                    'vel': (pos[j] - Q['pos']) / gap,
                    'mass': mass[j],
                    'mass_hist': self._push_hist(Q['mass_hist'], mass[j]),
                }
                succ[q] = ids[j]
                self.pending.pop(q, None)
            unmatched = [j for j in unmatched if ids[j] not in new_active]

        # ---- (d) dọn dẹp ----
        for q in list(self.pending.keys()):
            self.pending[q]['missed'] += 1
            if self.pending[q]['missed'] > MAX_SKIP_FRAMES:
                del self.pending[q]

        matched_prev = set(matched_curr.values())
        for p in prev_ids:
            if p in matched_prev:
                continue
            a = self.active[p]
            self.pending[p] = {'pos': a['pos'], 'vel': a['vel'],
                               'mass': a['mass'], 'mass_hist': a['mass_hist'],
                               'missed': 1}

        for j in unmatched:
            if ids[j] not in new_active:
                new_active[ids[j]] = {
                    'pos': pos[j], 'vel': np.zeros(3), 'mass': mass[j],
                    'mass_hist': [mass[j]],
                }

        # ---- (e) xác nhận phân bào: đủ tuổi + khoảng cách 2 con tăng ----
        still_pending = []
        for cand in self.div_pending:
            if cand['age'] == 0:
                # candidate vừa tạo ở bước (b) của CHÍNH step này — c1/c2 là
                # node của khung hiện tại, chưa bao giờ là "prev" nên chưa
                # có trong succ; chỉ tăng tuổi và chờ step sau
                cand['age'] = 1
                still_pending.append(cand)
                continue
            c1 = succ.get(cand['c1'])
            c2 = succ.get(cand['c2'])
            if c1 is None or c2 is None:
                self.n_div_rejected += 1          # một "con" biến mất → bỏ
                self.n_rej_lost += 1
                continue
            cand['c1'], cand['c2'] = c1, c2
            cand['age'] += 1
            if cand['age'] > DIV_CONFIRM_FRAMES:
                a1, a2 = new_active.get(c1), new_active.get(c2)
                if a1 is not None and a2 is not None:
                    d_now = float(np.linalg.norm(a1['pos'] - a2['pos']))
                    if d_now >= DIV_SEP_GROWTH * cand['d0']:
                        # XÁC NHẬN: ghi cạnh hoãn (mẹ t-1 → con thứ 2 t) —
                        # vẫn là cạnh liền khung vì mẹ/con1/con2 sinh cùng lượt
                        edges.append((cand['mother'], cand['c2_start']))
                        divisions.append((cand['mother'], cand['c2_start']))
                        self.n_div_confirmed += 1
                        continue
                self.n_div_rejected += 1          # không tách ra → merge-split giả
                self.n_rej_dyn += 1
                continue
            still_pending.append(cand)
        self.div_pending = still_pending

        self.active = new_active
        return nodes, edges, divisions, interp_nodes


# ---------- 4) CHẠY TOÀN BỘ TEST SET + CHẨN ĐOÁN ----------
def diagnose(rows):
    """Thống kê "sức khoẻ" submission — phát hiện sớm lỗi cấu trúc.
    Trả về chuỗi log để in cùng kết quả dataset."""
    nd = [r for r in rows if r['row_type'] == 'node']
    ed = [r for r in rows if r['row_type'] == 'edge']
    if not nd:
        return 'không có node'
    node_ids = {r['node_id'] for r in nd}
    has_in = {r['target_id'] for r in ed if r['target_id'] in node_ids}
    forks = {}
    for r in ed:
        if r['source_id'] in node_ids:
            forks[r['source_id']] = forks.get(r['source_id'], 0) + 1
    n_div = sum(1 for v in forks.values() if v >= 2)
    # track = thành phần liên thông yếu
    parent = {i: i for i in node_ids}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for r in ed:
        s, d = r['source_id'], r['target_id']
        if s in parent and d in parent:
            parent[find(s)] = find(d)
    from collections import Counter
    sizes = Counter(find(i) for i in node_ids)
    med_track = float(np.median(list(sizes.values()))) if sizes else 0.0
    n_no_in = sum(1 for r in nd if r['node_id'] not in has_in)
    t_lo = min(r['t'] for r in nd)
    t_hi = max(r['t'] for r in nd)
    frames = t_hi - t_lo + 1
    return (f"{len(nd)} node · {len(ed)} cạnh · {n_div} phân bào · "
            f"{n_no_in}/{len(nd)} node không cạnh vào ({100.0 * n_no_in / len(nd):.0f}%) · "
            f"{len(sizes)} track · trung vị {med_track:.0f} node/track · "
            f"{len(nd) / max(1, frames):.1f} node/khung · "
            f"phân bào xác nhận/từ chối: {TRK.n_div_confirmed}/{TRK.n_div_rejected} "
            f"(rớt mass: {TRK.n_rej_mass} · rớt động học: {TRK.n_rej_dyn} · "
            f"mất con: {TRK.n_rej_lost})")


if not os.path.isdir(TEST_DIR):
    hint = os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'không có /kaggle/input'
    raise FileNotFoundError(f'Không thấy {TEST_DIR} — hãy Add Input competition. /kaggle/input: {hint}')

test_folder_names = sorted(
    d.replace('.zarr', '') for d in os.listdir(TEST_DIR) if d.endswith('.zarr')
)
if not test_folder_names:
    raise RuntimeError(f'Không tìm thấy thư mục .zarr nào trong {TEST_DIR}: {os.listdir(TEST_DIR)[:10]}')
print(f'{len(test_folder_names)} dataset: {test_folder_names}')

all_rows = []
summary = []
STOPPED_EARLY = False

for folder_name in test_folder_names:
    zarr_path = os.path.join(TEST_DIR, folder_name + '.zarr')
    shape, dtype, chunk = read_zarr_meta(zarr_path)
    n_t = shape[0]

    TRK = Tracker()
    ds_rows = []
    n_nodes = n_edges = n_div = 0
    t0 = time.time()

    for t in range(n_t):
        if _over_time_budget():
            print(f'!! Gần hết ngân sách {TIME_LIMIT_HOURS}h — dừng tại {folder_name} t={t}')
            STOPPED_EARLY = True
            break
        try:
            vol = load_volume(zarr_path, t, shape, dtype, chunk)
        except (FileNotFoundError, ValueError) as e:
            print(f'  [{folder_name}] lỗi đọc khung {t}: {e} — bỏ qua')
            continue
        dets = detect_nodes(vol)
        nodes, edges, divisions, interp_nodes = TRK.step(dets, t)

        for nid, d in nodes:
            ds_rows.append({
                'dataset': folder_name, 'row_type': 'node', 'node_id': nid,
                't': t, 'z': d['z'], 'y': d['y'], 'x': d['x'],
                'source_id': -1, 'target_id': -1,
            })
        for nid, tk, d in interp_nodes:
            ds_rows.append({
                'dataset': folder_name, 'row_type': 'node', 'node_id': nid,
                't': tk, 'z': d['z'], 'y': d['y'], 'x': d['x'],
                'source_id': -1, 'target_id': -1,
            })
        for src, dst in edges:
            ds_rows.append({
                'dataset': folder_name, 'row_type': 'edge', 'node_id': -1,
                't': -1, 'z': -1, 'y': -1, 'x': -1,
                'source_id': src, 'target_id': dst,
            })
        n_nodes += len(nodes) + len(interp_nodes)
        n_edges += len(edges)
        n_div += len(divisions)

        if (t + 1) % 50 == 0 or t == n_t - 1:
            print(f'  [{folder_name}] {t + 1}/{n_t} khung · {n_nodes} node · '
                  f'{n_edges} cạnh · {n_div} phân bào · {time.time() - t0:.0f}s', flush=True)

    all_rows.extend(ds_rows)
    if DIAGNOSE:
        print(f'  [chẩn đoán {folder_name}] {diagnose(ds_rows)}', flush=True)
    summary.append((folder_name, n_nodes, n_edges, n_div))
    print(f'== {folder_name}: {n_nodes} nodes · {n_edges} edges · {n_div} phân bào '
          f'({time.time() - t0:.0f}s)', flush=True)
    if STOPPED_EARLY:
        break

if STOPPED_EARLY:
    print('CẢNH BÁO: dừng sớm do ngân sách thời gian — kết quả chỉ một phần!')

# ---------- 5) SOÁT BẰNG MẮT (tuỳ chọn) ----------
if RUN_PREVIEW and test_folder_names:
    import matplotlib.pyplot as plt

    zarr_path = os.path.join(TEST_DIR, test_folder_names[0] + '.zarr')
    shape, dtype, chunk = read_zarr_meta(zarr_path)
    vol = load_volume(zarr_path, 0, shape, dtype, chunk)
    dets = detect_nodes(vol)

    zc = shape[1] // 2
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax in axes:
        ax.imshow(vol[zc], cmap='gray')
        ax.set_xticks([])
        ax.set_yticks([])
    axes[0].set_title(f'{test_folder_names[0]} · t=0 · lát z={zc}')
    for d in dets:
        axes[1].plot(d['x'], d['y'], 'o', mfc='none', mec='lime', ms=9, mew=1.2)
    axes[1].set_title(f'phát hiện: {len(dets)} node (ver 3: P{PERCENTILE:g} + tách blob)')
    plt.tight_layout()
    plt.show()


In [ ]:
# ver 3 · cell 4 — XUẤT SUBMISSION
# Dán đè Cell 4 của notebook Kaggle.
# ============================================================

COLS = ['dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
submission = pd.DataFrame(all_rows, columns=COLS)
submission.index.name = 'id'
submission.to_csv('submission.csv')

n_nodes = int((submission['row_type'] == 'node').sum())
n_edges = int((submission['row_type'] == 'edge').sum())
n_div = sum(s[3] for s in summary)
print(f'Đã ghi submission.csv: {len(submission)} hàng '
      f'({n_nodes} node · {n_edges} cạnh · {n_div} phân bào)')
print(submission['row_type'].value_counts().to_string())
print(submission.head(8).to_string())
